# B5: Model Explainability

**Deliverable B Notebooks - Part B5**

Since this is an LLM-based extraction system (not a traditional ML model),
SHAP/feature importance do not directly apply. This notebook adapts
explainability concepts to the LLM context.

## 1. Introduction

For traditional ML models, explainability means understanding which features contribute most to predictions.
For LLM-based requirement extraction, explainability takes different forms:

1. **Source Traceability:** Exact SRS text that triggered extraction
2. **Classification Rationale:** Why FR vs NFR
3. **Confidence Assessment:** Which fields the model is uncertain about
4. **Clarification Reasoning:** Questions generated for ambiguous cases
5. **Structural Patterns:** Statistical relationships in the data

## 2. Load Test Fixtures

In [ ]:
import json, os

FIXTURE_DIR = os.path.join(os.getcwd(), 'datasets', 'pipeline-test')

with open(os.path.join(FIXTURE_DIR, 'confidence_input.json')) as f:
    conf_in = json.load(f)
with open(os.path.join(FIXTURE_DIR, 'confidence_output.json')) as f:
    conf_out = json.load(f)
with open(os.path.join(FIXTURE_DIR, 'clarify_input.json')) as f:
    clar_in = json.load(f)
with open(os.path.join(FIXTURE_DIR, 'clarify_output.json')) as f:
    clar_out = json.load(f)

print(f'Confidence input: {len(conf_in)} flag items')
print(f'Confidence output: {len(conf_out)} assessments')
print(f'Clarify input: {len(clar_in)} requirements')
print(f'Clarify output: {len(clar_out)} questions')

## 3. Confidence-based Explainability

The system flags uncertain requirement fields:
- **VAGUE:** Unclear or ambiguous language
- **INCOMPLETE:** Missing essential details
- **UNTESTABLE:** Acceptance criteria not verifiable

In [ ]:
from collections import Counter

flag_counts = Counter()
field_counts = Counter()
for doc in conf_out:
    for req_flags in doc.get('flags', {}).values():
        for flag in req_flags:
            flag_counts[flag.get('issue', 'unknown')] += 1
            field_counts[flag.get('field', 'unknown')] += 1

print('Issue types:')
for issue, count in flag_counts.most_common():
    print(f'  {issue}: {count}')

print()
print('Flagged fields:')
for field, count in field_counts.most_common():
    print(f'  {field}: {count}')

## 4. Clarification-based Explainability

For flagged requirements, the system generates targeted clarification
questions using Tree-of-Thought reasoning across 4 dimensions:
- FUNCTIONAL: What does the system actually do?
- SCOPE: What is in/out of scope?
- DEPENDENCY: What other features are required?
- COMPLIANCE: What standards/requirements apply?

In [ ]:
total_questions = sum(len(rq.get('questions', [])) for rq in clar_out.get('requirements', []))
print(f'Total clarification questions: {total_questions}')
for rq in clar_out.get('requirements', []):
    req_id = rq.get('requirement_id', 'unknown')
    nq = len(rq.get('questions', []))
    print(f'  {req_id}: {nq} questions')

## 5. Type Classification Signals

Type classification (FR vs NFR) is driven by semantic content patterns.

**Functional signals:**
- Action verbs: provide, create, display, allow, manage, update
- System interactions: page, content, data, file, report, form
- User capabilities: users can, users shall be able to

**Non-functional signals:**
- Performance: response time, throughput, availability, uptime
- Security: password, encryption, authentication, authorization
- Usability: intuitive, readable, accessible, color contrast
- Constraints: browser, platform, language, framework

## 6. Decision Boundary Analysis

In [ ]:
import os, statistics

PURE_BASE = '/Users/xd/Final_Project/Final Project/raw/datasets/PURE_output'
datasets_to_load = ['0000 - gamma j', '2007 - get real 0.2', '2010 - mashboot']

func_lengths, nfunc_lengths = [], []
func_acs, nfunc_acs = [], []

for name in datasets_to_load:
    path = os.path.join(PURE_BASE, name, 'requirements.json')
    with open(path) as f:
        reqs = json.load(f)
    for r in reqs:
        desc_len = len(r['description'])
        ac_count = len(r.get('acceptance_criteria', []))
        if r['type'] == 'functional':
            func_lengths.append(desc_len)
            func_acs.append(ac_count)
        else:
            nfunc_lengths.append(desc_len)
            nfunc_acs.append(ac_count)

print('Description length by type:')
print(f'  Functional: mean={statistics.mean(func_lengths):.0f}, median={statistics.median(func_lengths):.0f}')
print(f'  Non-functional: mean={statistics.mean(nfunc_lengths):.0f}, median={statistics.median(nfunc_lengths):.0f}')
print()
print('Acceptance criteria count by type:')
print(f'  Functional: mean={statistics.mean(func_acs):.1f}')
print(f'  Non-functional: mean={statistics.mean(nfunc_acs):.1f}')

## 7. Example Interpretations

In [ ]:
import json, os

GR_PATH = '/Users/xd/Final_Project/Final Project/raw/datasets/PURE_output/2007 - get real 0.2/requirements.json'
with open(GR_PATH) as f:
    gr = json.load(f)

for ex_id in ['REQ-002', 'REQ-004', 'REQ-006']:
    req = next((r for r in gr if r['requirement_id'] == ex_id), None)
    if req:
        print(f'\n=== {ex_id} ({req["type"].upper()}) ===')
        print(f'Title: {req["title"]}')
        print(f'Description: {req["description"][:150]}...')
        print(f'AC count: {len(req.get("acceptance_criteria", []))}')
        verse = req.get('source_location', {}).get('verse', [])
        if verse:
            print(f'Source: {verse[0][:100]}...')
        print()

## 8. Summary

### What we CAN explain:

1. **Source traceability:** Every requirement links back to exact SRS text
2. **Classification signals:** Keyword-based type classification rationale
3. **Confidence assessment:** Per-field confidence scores with specific flags
4. **Clarification reasoning:** Tree-of-Thought questions across 4 dimensions
5. **Structural patterns:** Descriptive/AC length statistics by type

### What we CANNOT explain:

1. **Internal LLM reasoning:** Opaque neural network weights
2. **Prompt sensitivity:** Small prompt changes can shift results
3. **Knowledge boundaries:** What the model knows vs hallucinates
4. **Deterministic guarantees:** LLM outputs are inherently probabilistic